# Data Modeling Techniques Comparison

This notebook demonstrates and compares different data modeling techniques:

1. **Data Vault 1.0** - Traditional hub, link, and satellite structure
2. **Data Vault 2.0** - Enhanced with business vault and point-in-time tables
3. **SCD (Slowly Changing Dimensions)** - Types 1, 2, 3, 4, and 6
4. **Big Fact Table** - Denormalized approach
5. **Star Schema** - Traditional dimensional modeling
6. **Snowflake Schema** - Normalized dimensional modeling

## Technology Stack
- **Database**: ClickHouse
- **Language**: Python
- **Libraries**: clickhouse-connect, pandas, matplotlib, seaborn

In [1]:
# Import required libraries
import sys
import os
sys.path.append('/app')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from datetime import datetime

# Import our custom modules
from src.utils.database import ClickHouseManager
from src.generators.data_generator import DataGenerator
from src.models.data_vault_1 import DataVault1Manager
from src.models.data_vault_2 import DataVault2Manager
from src.models.scd import SCDManager
from src.models.big_fact import BigFactManager
from src.models.star_schema import StarSchemaManager
from src.models.snowflake_schema import SnowflakeSchemaManager

# Configure plotting
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


## 1. Database Connection and Setup

In [2]:
# Initialize database connection
db = ClickHouseManager()

# Test connection
if db.test_connection():
    print("✓ ClickHouse connection successful!")
else:
    print("✗ ClickHouse connection failed!")
    raise Exception("Cannot proceed without database connection")

✓ Connected to ClickHouse at clickhouse:8123

✓ ClickHouse connection successful!


## 2. Data Generation

Let's generate sample e-commerce data for our demonstrations.

In [3]:
# Generate sample data
generator = DataGenerator()

# Generate datasets
print("Generating sample data...")
datasets = generator.generate_sample_data(scale=1000)  # 1000 customers, 2000 orders

print(f"Generated:")
for name, data in datasets.items():
    print(f"  - {name}: {len(data)} records")

Generating sample data...


Generating sample data (scale: 1000)

✓ Generated 1000 customers records

✓ Generated 500 products records

✓ Generated 100 suppliers records

✓ Generated 2000 orders records

✓ Generated 6061 order_lines records

✓ Generated 1461 time_dimension records

✓ Sample data generation completed!

Generated:
  - customers: 1000 records
  - products: 500 records
  - suppliers: 100 records
  - orders: 2000 records
  - order_lines: 6061 records
  - time_dimension: 1461 records


In [ ]:
# Explore the generated data
customers_df = pd.DataFrame(datasets['customers'])
products_df = pd.DataFrame(datasets['products'])
orders_df = pd.DataFrame(datasets['orders'])
order_lines_df = pd.DataFrame(datasets['order_lines'])

print("Sample Customer Data:")
print(customers_df.head())
print("\nSample Product Data:")
print(products_df.head())
print("\nSample Order Data:")
print(orders_df.head())

## 3. Data Vault 1.0 Implementation

Traditional Data Vault with Hubs, Links, and Satellites.

In [ ]:
# Load Data Vault 1.0
dv1_manager = DataVault1Manager(db)
dv1_manager.load_data()

print("Data Vault 1.0 loaded successfully!")

In [ ]:
# Query Data Vault 1.0 - Customer 360 view
customer_360 = dv1_manager.get_customer_360_view('CUST_000001')
print("Customer 360 View from Data Vault 1.0:")
print(customer_360)

# Show hub and satellite structure
hub_count = db.execute_query("SELECT count() FROM data_vault_1.hub_customer")
sat_count = db.execute_query("SELECT count() FROM data_vault_1.sat_customer")
link_count = db.execute_query("SELECT count() FROM data_vault_1.link_order_customer")

print(f"\nData Vault 1.0 Structure:")
print(f"  - Customer Hubs: {hub_count.result_rows[0][0]}")
print(f"  - Customer Satellites: {sat_count.result_rows[0][0]}")
print(f"  - Order-Customer Links: {link_count.result_rows[0][0]}")

## 4. Data Vault 2.0 Implementation

Enhanced Data Vault with Business Vault layer, Point-in-Time tables, and Multi-Active Satellites.

In [ ]:
# Load Data Vault 2.0
dv2_manager = DataVault2Manager(db)
dv2_manager.load_data()

print("Data Vault 2.0 loaded successfully!")

In [ ]:
# Query Data Vault 2.0 - Enhanced Customer view with Business Vault
customer_360_enhanced = dv2_manager.get_customer_360_with_business_vault('CUST_000001')
print("Enhanced Customer 360 View from Data Vault 2.0:")
print(customer_360_enhanced)

# Show Business Vault features
ref_count = db.execute_query("SELECT count() FROM data_vault_2.ref_product_category")
calc_sat_count = db.execute_query("SELECT count() FROM data_vault_2.sat_customer_metrics")
pit_count = db.execute_query("SELECT count() FROM data_vault_2.pit_customer")
multi_sat_count = db.execute_query("SELECT count() FROM data_vault_2.sat_customer_preferences")

print(f"\nData Vault 2.0 Business Vault Features:")
print(f"  - Reference Tables: {ref_count.result_rows[0][0]}")
print(f"  - Calculated Satellites: {calc_sat_count.result_rows[0][0]}")
print(f"  - Point-in-Time Tables: {pit_count.result_rows[0][0]}")
print(f"  - Multi-Active Satellites: {multi_sat_count.result_rows[0][0]}")

## 5. Slowly Changing Dimensions (SCD) Implementation

Demonstration of different SCD types.

In [ ]:
# Load SCD Demo
scd_manager = SCDManager(db)
scd_manager.load_data()

print("SCD Demo loaded successfully!")

In [ ]:
# Compare SCD types for the same customer
scd_comparison = scd_manager.compare_scd_types('CUST_000001')

print("SCD Types Comparison for Customer CUST_000001:")
print("\nType 1 (Overwrite):")
print(scd_comparison['type1'])

print("\nType 2 (Full History):")
for version in scd_comparison['type2']:
    print(f"  Version {version[14]}: {version[5]} - {version[6]}")

print("\nType 3 (Limited History):")
type3_data = scd_comparison['type3']
if type3_data:
    print(f"  Current Address: {type3_data[6]}")
    print(f"  Previous Address: {type3_data[7]}")

# Show SCD table counts
type1_count = db.execute_query("SELECT count() FROM scd_demo.dim_customer_type1")
type2_count = db.execute_query("SELECT count() FROM scd_demo.dim_customer_type2")
type3_count = db.execute_query("SELECT count() FROM scd_demo.dim_customer_type3")
type6_count = db.execute_query("SELECT count() FROM scd_demo.dim_customer_type6")

print(f"\nSCD Table Counts:")
print(f"  - Type 1: {type1_count.result_rows[0][0]}")
print(f"  - Type 2: {type2_count.result_rows[0][0]}")
print(f"  - Type 3: {type3_count.result_rows[0][0]}")
print(f"  - Type 6: {type6_count.result_rows[0][0]}")

## 6. Big Fact Table Implementation

Denormalized approach with all data in one large table.

In [ ]:
# Load Big Fact Table
big_fact_manager = BigFactManager(db)
big_fact_manager.load_data()

print("Big Fact Table loaded successfully!")

In [ ]:
# Query Big Fact Table
customer_summary = big_fact_manager.get_customer_summary('CUST_000001')
print("Customer Summary from Big Fact Table:")
print(customer_summary)

performance_metrics = big_fact_manager.get_performance_metrics()
print("\nOverall Performance Metrics:")
print(performance_metrics)

# Show table size
big_fact_count = db.execute_query("SELECT count() FROM big_fact.big_fact_sales")
print(f"\nBig Fact Table Records: {big_fact_count.result_rows[0][0]:,}")

## 7. Star Schema Implementation

Traditional dimensional modeling with separate dimension and fact tables.

In [ ]:
# Load Star Schema
star_manager = StarSchemaManager(db)
star_manager.load_data()

print("Star Schema loaded successfully!")

In [ ]:
# Query Star Schema
top_customers = star_manager.get_sales_by_dimension('customer', 5)
print("Top 5 Customers by Sales (Star Schema):")
for customer in top_customers:
    print(f"  {customer[0]}: ${customer[1]:,.2f}")

top_products = star_manager.get_sales_by_dimension('product', 5)
print("\nTop 5 Products by Sales (Star Schema):")
for product in top_products:
    print(f"  {product[0]}: ${product[1]:,.2f}")

# Show dimension table counts
dim_counts = {
    'Time': db.execute_query("SELECT count() FROM star_schema.dim_time").result_rows[0][0],
    'Customer': db.execute_query("SELECT count() FROM star_schema.dim_customer").result_rows[0][0],
    'Product': db.execute_query("SELECT count() FROM star_schema.dim_product").result_rows[0][0],
    'Geography': db.execute_query("SELECT count() FROM star_schema.dim_geography").result_rows[0][0],
    'Fact Sales': db.execute_query("SELECT count() FROM star_schema.fact_sales").result_rows[0][0]
}

print(f"\nStar Schema Table Counts:")
for table, count in dim_counts.items():
    print(f"  - {table}: {count:,}")

## 8. Snowflake Schema Implementation

Normalized dimensional modeling with hierarchical dimensions.

In [ ]:
# Load Snowflake Schema
snowflake_manager = SnowflakeSchemaManager(db)
snowflake_manager.load_data()

print("Snowflake Schema loaded successfully!")

In [ ]:
# Query Snowflake Schema - Hierarchical report
hierarchical_report = snowflake_manager.get_hierarchical_sales_report()
print("Hierarchical Sales Report (Snowflake Schema):")
print("Category L1 | Category L2 | Category L3 | Brand | Product | Sales")
print("-" * 80)
for row in hierarchical_report[:10]:
    print(f"{row[0][:10]} | {row[1][:10]} | {row[2][:10]} | {row[3][:8]} | {row[4][:15]} | ${row[5]:,.0f}")

# Show normalized dimension counts
snowflake_counts = {
    'Years': db.execute_query("SELECT count() FROM snowflake_schema.dim_year").result_rows[0][0],
    'Quarters': db.execute_query("SELECT count() FROM snowflake_schema.dim_quarter").result_rows[0][0],
    'Months': db.execute_query("SELECT count() FROM snowflake_schema.dim_month").result_rows[0][0],
    'Dates': db.execute_query("SELECT count() FROM snowflake_schema.dim_date").result_rows[0][0],
    'Continents': db.execute_query("SELECT count() FROM snowflake_schema.dim_continent").result_rows[0][0],
    'Countries': db.execute_query("SELECT count() FROM snowflake_schema.dim_country").result_rows[0][0],
    'States': db.execute_query("SELECT count() FROM snowflake_schema.dim_state").result_rows[0][0],
    'Cities': db.execute_query("SELECT count() FROM snowflake_schema.dim_city").result_rows[0][0],
    'Brands': db.execute_query("SELECT count() FROM snowflake_schema.dim_brand").result_rows[0][0],
    'Category L1': db.execute_query("SELECT count() FROM snowflake_schema.dim_category_l1").result_rows[0][0],
    'Category L2': db.execute_query("SELECT count() FROM snowflake_schema.dim_category_l2").result_rows[0][0],
    'Category L3': db.execute_query("SELECT count() FROM snowflake_schema.dim_category_l3").result_rows[0][0],
    'Fact Sales': db.execute_query("SELECT count() FROM snowflake_schema.fact_sales").result_rows[0][0]
}

print(f"\nSnowflake Schema Table Counts:")
for table, count in snowflake_counts.items():
    print(f"  - {table}: {count:,}")

## 9. Performance Comparison

Let's compare query performance across different modeling techniques.

In [ ]:
def measure_query_performance(query, description):
    """Measure query execution time."""
    start_time = time.time()
    try:
        result = db.execute_query(query)
        end_time = time.time()
        execution_time = (end_time - start_time) * 1000  # Convert to milliseconds
        row_count = len(result.result_rows) if result.result_rows else 0
        return execution_time, row_count, True
    except Exception as e:
        return 0, 0, False

# Define standard queries for each schema
queries = {
    "Customer Sales Summary": {
        "Big Fact": """
            SELECT customer_id, customer_full_name, 
                   sum(line_total) as total_sales,
                   count() as order_count
            FROM big_fact.big_fact_sales 
            GROUP BY customer_id, customer_full_name 
            ORDER BY total_sales DESC LIMIT 100
        """,
        "Star Schema": """
            SELECT dc.customer_id, dc.full_name,
                   sum(fs.line_total) as total_sales,
                   count() as order_count
            FROM star_schema.fact_sales fs
            JOIN star_schema.dim_customer dc ON fs.customer_key = dc.customer_key
            GROUP BY dc.customer_id, dc.full_name
            ORDER BY total_sales DESC LIMIT 100
        """,
        "Snowflake Schema": """
            SELECT dc.customer_id, dc.full_name,
                   sum(fs.line_total) as total_sales,
                   count() as order_count
            FROM snowflake_schema.fact_sales fs
            JOIN snowflake_schema.dim_customer dc ON fs.customer_key = dc.customer_key
            GROUP BY dc.customer_id, dc.full_name
            ORDER BY total_sales DESC LIMIT 100
        """
    },
    "Product Category Analysis": {
        "Big Fact": """
            SELECT product_category_l1, product_brand,
                   sum(line_total) as total_sales,
                   avg(profit_margin) as avg_margin
            FROM big_fact.big_fact_sales
            GROUP BY product_category_l1, product_brand
            ORDER BY total_sales DESC LIMIT 50
        """,
        "Star Schema": """
            SELECT dp.category, dp.brand,
                   sum(fs.line_total) as total_sales,
                   avg(fs.profit_margin) as avg_margin
            FROM star_schema.fact_sales fs
            JOIN star_schema.dim_product dp ON fs.product_key = dp.product_key
            GROUP BY dp.category, dp.brand
            ORDER BY total_sales DESC LIMIT 50
        """,
        "Snowflake Schema": """
            SELECT dc1.category_l1_name, db.brand_name,
                   sum(fs.line_total) as total_sales,
                   avg(fs.profit_margin) as avg_margin
            FROM snowflake_schema.fact_sales fs
            JOIN snowflake_schema.dim_product dp ON fs.product_key = dp.product_key
            JOIN snowflake_schema.dim_brand db ON dp.brand_key = db.brand_key
            JOIN snowflake_schema.dim_category_l3 dc3 ON dp.category_l3_key = dc3.category_l3_key
            JOIN snowflake_schema.dim_category_l2 dc2 ON dc3.category_l2_key = dc2.category_l2_key
            JOIN snowflake_schema.dim_category_l1 dc1 ON dc2.category_l1_key = dc1.category_l1_key
            GROUP BY dc1.category_l1_name, db.brand_name
            ORDER BY total_sales DESC LIMIT 50
        """
    }
}

# Run performance tests
performance_results = []

for query_name, schema_queries in queries.items():
    print(f"\nTesting: {query_name}")
    for schema_name, query in schema_queries.items():
        exec_time, row_count, success = measure_query_performance(query, f"{schema_name} - {query_name}")
        if success:
            print(f"  {schema_name}: {exec_time:.2f}ms ({row_count} rows)")
            performance_results.append({
                'Query': query_name,
                'Schema': schema_name,
                'Execution_Time_ms': exec_time,
                'Row_Count': row_count
            })
        else:
            print(f"  {schema_name}: FAILED")

# Create performance DataFrame
perf_df = pd.DataFrame(performance_results)
print("\nPerformance Results Summary:")
print(perf_df)

In [ ]:
# Visualize performance comparison
if not perf_df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Execution time comparison
    pivot_time = perf_df.pivot(index='Query', columns='Schema', values='Execution_Time_ms')
    pivot_time.plot(kind='bar', ax=ax1, title='Query Execution Time by Schema')
    ax1.set_ylabel('Execution Time (ms)')
    ax1.legend(title='Schema Type')
    ax1.tick_params(axis='x', rotation=45)
    
    # Average execution time by schema
    avg_time = perf_df.groupby('Schema')['Execution_Time_ms'].mean()
    avg_time.plot(kind='bar', ax=ax2, title='Average Query Execution Time')
    ax2.set_ylabel('Average Execution Time (ms)')
    ax2.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print("No performance data available for visualization.")

## 10. Storage Comparison

Let's compare storage requirements across different schemas.

In [ ]:
# Get storage information for each database
databases = ['data_vault_1', 'data_vault_2', 'scd_demo', 'big_fact', 'star_schema', 'snowflake_schema']
storage_info = []

for database in databases:
    table_count = db.get_table_count(database)
    row_count = db.get_row_count(database)
    size_mb = db.get_database_size(database)
    
    storage_info.append({
        'Database': database,
        'Tables': table_count,
        'Total_Rows': row_count,
        'Size_MB': size_mb
    })

storage_df = pd.DataFrame(storage_info)
print("Storage Comparison:")
print(storage_df)

# Visualize storage comparison
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Number of tables
storage_df.plot(x='Database', y='Tables', kind='bar', ax=ax1, title='Number of Tables')
ax1.tick_params(axis='x', rotation=45)

# Total rows
storage_df.plot(x='Database', y='Total_Rows', kind='bar', ax=ax2, title='Total Rows', color='orange')
ax2.tick_params(axis='x', rotation=45)

# Storage size
storage_df.plot(x='Database', y='Size_MB', kind='bar', ax=ax3, title='Storage Size (MB)', color='green')
ax3.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 11. Complexity Analysis

Let's analyze the complexity of queries across different schemas.

In [ ]:
# Complexity analysis based on JOIN count and query structure
complexity_analysis = {
    'Big Fact Table': {
        'JOINs_Required': 0,
        'Query_Complexity': 'Low',
        'Maintenance_Complexity': 'Medium',
        'Storage_Efficiency': 'Low',
        'Query_Performance': 'High',
        'Flexibility': 'Low'
    },
    'Star Schema': {
        'JOINs_Required': 1-3,
        'Query_Complexity': 'Low-Medium',
        'Maintenance_Complexity': 'Medium',
        'Storage_Efficiency': 'Medium',
        'Query_Performance': 'High',
        'Flexibility': 'Medium'
    },
    'Snowflake Schema': {
        'JOINs_Required': 3-6,
        'Query_Complexity': 'High',
        'Maintenance_Complexity': 'High',
        'Storage_Efficiency': 'High',
        'Query_Performance': 'Medium',
        'Flexibility': 'High'
    },
    'Data Vault 1.0': {
        'JOINs_Required': 2-4,
        'Query_Complexity': 'Medium-High',
        'Maintenance_Complexity': 'Low',
        'Storage_Efficiency': 'Medium',
        'Query_Performance': 'Medium',
        'Flexibility': 'High'
    },
    'Data Vault 2.0': {
        'JOINs_Required': 3-5,
        'Query_Complexity': 'High',
        'Maintenance_Complexity': 'Low',
        'Storage_Efficiency': 'Medium',
        'Query_Performance': 'Medium',
        'Flexibility': 'Very High'
    },
    'SCD': {
        'JOINs_Required': 1-2,
        'Query_Complexity': 'Medium',
        'Maintenance_Complexity': 'High',
        'Storage_Efficiency': 'Variable',
        'Query_Performance': 'High',
        'Flexibility': 'Medium'
    }
}

print("Data Modeling Techniques Comparison:")
print("=" * 80)
for technique, characteristics in complexity_analysis.items():
    print(f"\n{technique}:")
    for char, value in characteristics.items():
        print(f"  {char.replace('_', ' ')}: {value}")

## 12. Use Case Recommendations

Based on our analysis, here are the recommended use cases for each modeling technique.

In [ ]:
recommendations = {
    'Big Fact Table': {
        'Best_For': [
            'Simple reporting requirements',
            'High-performance analytics',
            'Limited dimensional changes',
            'Small to medium datasets'
        ],
        'Avoid_When': [
            'Storage is a concern',
            'Complex dimensional hierarchies needed',
            'Frequent schema changes',
            'Multiple data sources'
        ]
    },
    'Star Schema': {
        'Best_For': [
            'Traditional BI and reporting',
            'OLAP cubes',
            'User-friendly query patterns',
            'Balanced performance and storage'
        ],
        'Avoid_When': [
            'Complex dimensional hierarchies',
            'Real-time data integration',
            'Extreme performance requirements',
            'Highly normalized source systems'
        ]
    },
    'Snowflake Schema': {
        'Best_For': [
            'Complex dimensional hierarchies',
            'Storage optimization',
            'Highly normalized environments',
            'Flexible reporting requirements'
        ],
        'Avoid_When': [
            'Simple reporting needs',
            'Performance is critical',
            'Limited technical expertise',
            'Frequent ad-hoc queries'
        ]
    },
    'Data Vault 1.0': {
        'Best_For': [
            'Enterprise data warehouses',
            'Audit and compliance requirements',
            'Multiple source systems',
            'Long-term data retention'
        ],
        'Avoid_When': [
            'Simple data models',
            'Limited technical resources',
            'Immediate reporting needs',
            'Small-scale projects'
        ]
    },
    'Data Vault 2.0': {
        'Best_For': [
            'Agile data warehousing',
            'Big data environments',
            'Real-time analytics',
            'Machine learning pipelines'
        ],
        'Avoid_When': [
            'Traditional BI tools only',
            'Simple data models',
            'Limited development resources',
            'Static reporting requirements'
        ]
    },
    'SCD (Slowly Changing Dimensions)': {
        'Best_For': [
            'Historical trend analysis',
            'Customer behavior tracking',
            'Regulatory compliance',
            'Point-in-time reporting'
        ],
        'Avoid_When': [
            'Rapidly changing dimensions',
            'Storage is limited',
            'Simple current-state reporting',
            'Real-time requirements'
        ]
    }
}

print("Data Modeling Technique Recommendations:")
print("=" * 80)

for technique, rec in recommendations.items():
    print(f"\n{technique}:")
    print("  Best For:")
    for item in rec['Best_For']:
        print(f"    • {item}")
    print("  Avoid When:")
    for item in rec['Avoid_When']:
        print(f"    • {item}")

## 13. Summary and Conclusions

This comprehensive comparison demonstrates the strengths and trade-offs of different data modeling techniques.

In [ ]:
# Summary statistics
print("Data Modeling Techniques Comparison Summary")
print("=" * 50)

print("\n📊 Storage Efficiency (Best to Worst):")
storage_sorted = storage_df.sort_values('Size_MB')
for _, row in storage_sorted.iterrows():
    print(f"  {row['Database']}: {row['Size_MB']:.1f} MB")

if not perf_df.empty:
    print("\n⚡ Query Performance (Fastest to Slowest):")
    avg_perf = perf_df.groupby('Schema')['Execution_Time_ms'].mean().sort_values()
    for schema, time_ms in avg_perf.items():
        print(f"  {schema}: {time_ms:.1f} ms average")

print("\n🏗️ Complexity Levels:")
complexity_levels = {
    'Low': ['Big Fact Table'],
    'Medium': ['Star Schema', 'SCD'],
    'High': ['Data Vault 1.0', 'Snowflake Schema'],
    'Very High': ['Data Vault 2.0']
}

for level, techniques in complexity_levels.items():
    print(f"  {level}: {', '.join(techniques)}")

print("\n🎯 Key Takeaways:")
print("  • Big Fact Tables: Best for simple, high-performance scenarios")
print("  • Star Schema: Optimal balance for traditional BI")
print("  • Snowflake Schema: Best for complex hierarchies and storage optimization")
print("  • Data Vault 1.0: Excellent for enterprise data warehouses")
print("  • Data Vault 2.0: Future-proof for agile and big data environments")
print("  • SCD: Essential for historical and compliance requirements")

print("\n✅ Demo completed successfully!")
print(f"Generated and loaded data across {len(databases)} different modeling approaches.")